In [ ]:
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random

# Create Data

To simulate a timeseries, let's take a mean value and add some random noise with a chosen standard deviation.
For the purpose of this challenge, we have provided plausible values for each sensor.

In [ ]:
sensorNames = ['AtmP','Temp','Airtight','H2OC']
sensorCenterLines = [989.21,9.45,1216.02,9.64]
standardDeviation = [8.35,8.42,39.98,4.23]

For our use case, we need data that contains the following information:
- `sensor` - sensor name
- `reading` - the value sent by the sensor
- `timeStamp` - the time that the value has been sent

We want to create our data as a function of time. To create 100 datapoints (for 100 consecutive seconds), with a value for each sensor for every second, we can execute the following code block. Take your time to read through and understand what is happening.

In [ ]:
l_sensors = []
time_stamp_start = datetime.datetime(2022,11,25,9,0,0)
time_step = datetime.timedelta(seconds=1)
for i in range(100):
    for pos in range(0,4):
        l_sensors.append({
            'sensor': sensorNames[pos],
            'reading':np.random.normal(loc=sensorCenterLines[pos], scale=standardDeviation[pos]),
            'timeStamp':time_stamp_start + i* time_step
        })

df_sensors = pd.DataFrame(l_sensors)
df_sensors.head(10)

Now that we have the data let's aggregate the values by timestamp (index) and by sensor (columns). You can use `pandas.pivot_table` to do this.

In [ ]:
df_data = pd.pivot_table(df_sensors, values='reading', index="timeStamp", columns="sensor", aggfunc='mean')
df_data.head(10)

Let's plot the different sensors values against time.

In [ ]:
df_data.plot(subplots = True);

# Introducing Latency

In real life, sensors are not synchronised and they sometimes exhibit some latency when sending signals. We can simulate this latency by sampling from a probability distribution. In the following code block we can see 1000 values sampled from a normal distribution with mean of 0 and a standard deviation of 0.2.


In [ ]:
mu, sigma = 0, 0.2  # mean and standard deviation
s = abs(random.gauss(0,0.2)) # absolute because we can't have negative latency

s = []
for _ in range(1000):
    s.append(abs(random.gauss(0, 0.2)))

plt.hist(s)
plt.title("Possible latency distribution in sec")
plt.show()

We can add this latency component to the `timeStamp` when we create the synthetic data. This will create a unique latency for each sensor, for each timestamp.

In [ ]:
l_sensors_latency = []
for i in range(1500):
    for pos in range(0,4):
        l_sensors_latency.append({
            'sensor': sensorNames[pos],
            'reading': np.random.normal(loc=sensorCenterLines[pos], scale=standardDeviation[pos]),
            'timeStamp': time_stamp_start + i* time_step + datetime.timedelta(seconds=abs(random.gauss(0,0.2)))
        })

df_sensors_latency = pd.DataFrame(l_sensors_latency)
df_sensors_latency.head(10)

❗ The output dataframe above is the data format you will be working with in the next challenge. You will just be starting from a CSV!

To visualize, we can pivot the data and view for each sensor:

In [ ]:
df_data_latency = pd.pivot_table(df_sensors_latency,
                                  values='reading',
                                  index='timeStamp',
                                  columns='sensor',
                                  aggfunc='mean')
df_data_latency

In [ ]:
df_data_latency.plot(subplots=True);

💡 We can not gather the data continuously anymore because the data is not available for each sensor at the same time!

In that case it is better to group the data and in **fixed windows** (of 15 seconds for example) and take the mean over each window, for each sensor.

Let's resample the timestamp index with Pandas. 

In [ ]:
df_data_latency = df_data_latency.resample('15s').mean()
df_data_latency.head()

In [ ]:
df_data_latency.plot(subplots=True);

# 🏁 Finishing up

You should now have a deeper understanding of the data that we'll be working with for the next challenge!



🧪 Make sure to run:

```bash
make test
```

And git add, commit, and push your code to GitHub before continuing to the next challenge 🚀